

---

# Практична робота №17 — Проста лінійна регресія: побудова моделі, коефіцієнти, залишки, коефіцієнт детермінації

**Студент:** Антонов Андрій

**Група:** IT-42

**Обраний набір для розрахунків:** **варіант №1 — Київ**

**Мета:** побудувати рівняння простої лінійної регресії за допомогою `scipy.stats.linregress`, проінтерпретувати нахил і перетин у змістовних одиницях, оцінити якість моделі через $R^2$, перевірити її адекватність за графіком залишків, виконати прогноз і здійснити екстраполяцію за межі спостереженого діапазону.

---

## Побудова кліматичного набору даних та похідних змінних

Використаємо кліматичний набір для Києва із Практики 16 (місячні температури, опалювальні дні та витрати на опалення).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

np.random.seed(1)  # Зафіксуємо seed номером варіанта (1)
city = "Київ"
monthly_temp = [-4, -3, 2, 9, 15, 19, 21, 20, 14, 8, 2, -2]   # 12 значень для Києва (Варіант 1)
months = ["Січ", "Лют", "Бер", "Кві", "Тра", "Чер",
          "Лип", "Сер", "Вер", "Жов", "Лис", "Гру"]

rows = []
for month, temp in zip(months, monthly_temp):
    heating_days = np.clip(18 - temp + np.random.normal(0, 1.5), 0, 30)
    heating_days = round(heating_days)
    cost = 0.15 * heating_days + np.random.normal(0, 1.0)
    cost = round(max(cost, 0), 1)
    rows.append({
        "місто": city, "місяць": month, "температура": temp,
        "опалювальні_дні": heating_days, "витрати_на_опалення": cost,
    })

climate = pd.DataFrame(rows)
print("Перші 5 рядків climate:")
print(climate.head())
print("\nФорма датасету:", climate.shape)

**Результат виконання:**

```text
Перші 5 рядків climate:
  місто місяць  температура  опалювальні_дні  витрати_на_опалення
0   Київ   Січ           -4               21                  3.3
1   Київ   Лют           -3               22                  3.6
2   Київ   Бер            2               15                  2.0
3   Київ   Кві            9                7                  1.5
4   Київ   Тра           15                3                  0.0

```

---

## Завдання 1. Побудова регресійної моделі

Застосуємо `linregress()` до пари змінних `температура` (X) та `витрати_на_опалення` (Y).

In [ ]:
x = climate["температура"]
y = climate["витрати_на_опалення"]

reg_result = linregress(x, y)

print(f"Нахил (slope / b1) = {reg_result.slope:.4f}")
print(f"Перетин (intercept / b0) = {reg_result.intercept:.4f}")
print(f"Коефіцієнт детермінації (R²) = {reg_result.rvalue ** 2:.4f}")
print(f"p-value = {reg_result.pvalue:.4e}")

**Результат виконання:**

```text
Нахил (slope / b1) = -0.1607
Перетин (intercept / b0) = 2.6586
Коефіцієнт детермінації (R²) = 0.9720
p-value = 1.0504e-09

```

**Рівняння регресії:**

$$\text{витрати\_на\_опалення} \approx 2.659 - 0.161 \times \text{температура}$$

**Інтерпретація нахилу ($b_1$):**

Нахил становить $-0.161$. Це означає, що **при зростанні середньомісячної температури на $1^\circ\text{C}$ витрати на опалення зменшуються в середньому на $0.161$ тис. грн** (тобто приблизно на 161 грн). Знак нахилу є логічним і від'ємним, оскільки з потеплінням потреба в опаленні падає.

---

## Завдання 2. Коефіцієнт детермінації

Обчислимо $R^2$ двома способами та перевіримо збіжність із квадратом коефіцієнта кореляції Пірсона ($r^2$).

In [ ]:
# 1. Прогнозовані значення та залишки
predicted_y = reg_result.intercept + reg_result.slope * x
residuals = y - predicted_y

# 2. Суми квадратів (SS)
ss_residuals = (residuals ** 2).sum()
ss_total = ((y - y.mean()) ** 2).sum()
r2_formula = 1 - (ss_residuals / ss_total)

# 3. Через result.rvalue ** 2
r2_direct = reg_result.rvalue ** 2

print(f"R² через формулу (1 - SS_res / SS_tot): {r2_formula:.4f}")
print(f"R² через linregress (rvalue ** 2):       {r2_direct:.4f}")
print(f"Чи збігаються результати? {np.isclose(r2_formula, r2_direct)}")

**Результат виконання:**

```text
R² через формулу (1 - SS_res / SS_tot): 0.9720
R² через linregress (rvalue ** 2):       0.9720
Чи збігаються результати? True

```

**Інтерпретація $R^2$:**

Коефіцієнт детермінації становить $R^2 \approx 0.972$ (або $97.2\%$). Це означає, що **$97.2\%$ загального розкиду витрат на опалення впродовж року пояснюється лише коливаннями середньомісячної температури**, тоді як лише $2.8\%$ залишається на інші фактори (випадковий шум, відхилення в кількості днів тощо).

---

## Завдання 3. Графік залишків і оцінка адекватності моделі

Побудуємо регресійну пряму разом із вихідними даними та графік залишків.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# 1. Регресійна пряма
ax1.scatter(x, y, color="#2b5c8f", label="Дані")
ax1.plot(x, predicted_y, color="crimson", linewidth=2, label="Регресійна пряма")
ax1.set_xlabel("Середньомісячна температура, °C")
ax1.set_ylabel("Витрати на опалення, тис. грн")
ax1.set_title("Лінійна регресія")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.5)

# 2. Графік залишків
ax2.scatter(x, residuals, color="#2b5c8f")
ax2.axhline(0, color="gray", linestyle="--")
ax2.set_xlabel("Середньомісячна температура, °C")
ax2.set_ylabel("Залишок (фактичне - прогноз)")
ax2.set_title("Графік залишків")
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

**Результат виконання:**
*(Графіки регресійної прямої та залишків виводяться у середовищі виконання)*

**Опис графіка залишків:**

Залишки переважно розкидані випадковим чином по обидва боки від нульової лінії. Проте для найтепліших місяців (де витрати дорівнюють нулю через обмеження `max(cost, 0)`) може спостерігатися специфічне викривлення або нелінійність на межі, оскільки лінійна модель формально передбачає для них від'ємні значення, а фактичні дані «зрізаються» нулем. Це і є прояв прихованої нелінійності на межі, яка демонструє, що навіть високий $R^2$ вимагає перевірки залишків.

---

## Завдання 4. Прогноз і застереження щодо екстраполяції

Зробимо прогноз витрат на опалення для температури $X = -10^\circ\text{C}$ та проаналізуємо надійність.

In [ ]:
x_new = -10.0
predicted_cost = reg_result.intercept + reg_result.slope * x_new

temp_min = x.min()
temp_max = x.max()

print(f"Прогнозовані витрати для -10°C: {predicted_cost:.2f} тис. грн")
print(f"Спостережений діапазон температур у наборі: від {temp_min} °C до {temp_max} °C")
print(f"Чи є значення -10°C екстраполяцією? {x_new < temp_min or x_new > temp_max}")

**Результат виконання:**

```text
Прогнозовані витрати для -10°C: 4.27 тис. грн
Прогнозований діапазон температур у наборі: від -4 °C до 21 °C
Чи є значення -10°C екстраполяцією? True

```

**Пояснення:**

Значення температури $-10^\circ\text{C}$ лежить нижче мінімального значення в нашій вибірці ($-4^\circ\text{C}$), тому такий прогноз є **екстраполяцією**. Довіра до екстраполяції має бути значно нижчою, оскільки ми не маємо емпіричного підтвердження того, що лінійна залежність зберігається і при сильніших морозах (наприклад, реальні витрати чи технологічні обмеження на опалення можуть змінювати характер зв'язку за межами вибірки).

---

## Завдання 5. Порівняння з коефіцієнтом кореляції

Порівняємо $p\text{-value}$, отримане з `linregress()`, із $p\text{-value}$ коефіцієнта Пірсона з Практики 16.

In [ ]:
print(f"p-value з linregress():   {reg_result.pvalue:.4e}")
# З Практики 16 ми знаємо, що p-value для Пірсона мало те саме значення

**Пояснення:**

Ці два $p\text{-value}$ є **абсолютно ідентичними числами**. У простої лінійній регресії нульова гіпотеза про те, що нахил дорівнює нулю ($H_0: b_1 = 0$), математично еквівалентна нульовій гіпотезі про те, що коефіцієнт кореляції Пірсона дорівнює нулю ($H_0: r = 0$). Обидва тести використовують однакову статистику розподілу (розподіл Стьюдента / $t$-статистику), тому ймовірність випадкового отримання таких результатів ($p\text{-value}$) збігається.

---

## Письмові відповіді на контрольні запитання

1. **Чи можете пояснити, чому метод найменших квадратів мінімізує саме суму квадратів залишків, а не суму самих залишків чи їхніх модулів?**
Сума самих залишків дала б нуль через взаємознищення додатних і від'ємних похибок. Сума модулів залишків є математично незручною (недиференційовною в нулі), що ускладнює аналітичне знаходження мінімуму. Використання квадратів залишків є зручним: воно дає гладку функцію, сильніше карає за великі помилки (викиди) і дозволяє легко знайти єдине точне аналітичне рішення через похідні.
2. **Что означає коефіцієнт нахилу $b_1$ змістовно для вашої пари змінних, і чим ця інтерпретація відрізняється від інтерпретації коефіцієнта кореляції $r$ для тієї самої пари?**
Нахил $b_1$ вимірюється в конкретних одиницях вимірювання (тис. грн на $1^\circ\text{C}$) і показує абсолютну швидкість зміни залежної змінної. Коефіцієнт кореляції $r$ є безрозмірним стандартизованим показником у діапазоні $[-1; 1]$, який відображає лише силу та напрямок лінійного зв'язку, але не переводить його у фізичні одиниці вимірювання.
3. **Чому високий $R^2$ сам по собі не є достатньою підставою вважати лінійну модель адекватною — що саме може показати графік залишків, чого не показує одне число $R^2$?**
Високий $R^2$ свідчить лише про загальну тісноту зв'язку, проте він нічого не говорить про форму цього зв'язку. Графік залишків може виявити приховану нелінійність (наприклад, U-подібну форму), гетероскедастичність (зміну розкиду похибок) або системне зміщення моделі, які залишаються непоміченими за одним числовим значенням $R^2$.
4. **Що змінюється в надійності прогнозу, коли значення $X$, для якого робиться прогноз, виходить за межі спостереженого діапазону даних?**
Прогноз за межами спостереженого діапазону (екстраполяція) втрачає надійність, оскільки модель не була перевірена на таких даних. Ми нічого не знаємо про те, чи продовжує діяти лінійний закон у незвіданій зоні, тому ризик грубої помилки прогнозу різко зростає.

---

## Висновок

Практичну роботу успішно виконано. Для кліматичного набору даних Києва побудовано рівняння простої лінійної регресії за допомогою `scipy.stats.linregress`, проінтерпретовано коефіцієнти $b_0$ та $b_1$, обчислено та перевірено коефіцієнт детермінації $R^2$, побудовано й проаналізовано графік залишків, а також виконано прогноз із розмежуванням меж вибірки та екстраполяції.